In [ ]:
!pip install -q ultralytics

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from ultralytics import YOLO

# Load the YOLO model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")

In [ ]:
import cv2
from IPython.display import display, clear_output
from PIL import Image

video = cv2.VideoCapture("People.mp4")

# Test only 30 frames
for i in range(30):

    ret, frame = video.read()

    if not ret:
        print("Could not read video.")
        break

    # Detect only people (class 0)
    results = model(
        frame,
        classes=[0],
        verbose=False
    )

    # Draw bounding boxes
    annotated_frame = results[0].plot()

    # Convert BGR → RGB
    annotated_frame = cv2.cvtColor(
        annotated_frame,
        cv2.COLOR_BGR2RGB
    )

    clear_output(wait=True)
    display(Image.fromarray(annotated_frame))

video.release()

print("✅ 30-frame detection test completed!")

In [ ]:
import cv2
from ultralytics import YOLO

# Open the video
video = cv2.VideoCapture("People.mp4")

# Process only the first 10 seconds for our tracking test
fps = video.get(cv2.CAP_PROP_FPS)
max_frames = int(fps * 10)

print("Testing tracking for 10 seconds...")

for frame_number in range(max_frames):

    ret, frame = video.read()

    if not ret:
        break

    # YOLO + ByteTrack
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0],
        conf=0.4,
        verbose=False
    )

    # Draw boxes and tracking IDs
    annotated_frame = results[0].plot()

    # Display every 10th frame
    if frame_number % 10 == 0:

        annotated_frame = cv2.cvtColor(
            annotated_frame,
            cv2.COLOR_BGR2RGB
        )

        clear_output(wait=True)
        display(Image.fromarray(annotated_frame))

video.release()

print("✅ 10-second tracking test completed!")

In [ ]:
import cv2

# Open the input video
video = cv2.VideoCapture("People.mp4")

# Get video information
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv2.CAP_PROP_FPS)

# Create output video
output = cv2.VideoWriter(
    "final_people_tracking.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Convert ByteTrack IDs into simple IDs
id_mapping = {}
next_person_id = 1

frame_count = 0

while True:

    ret, frame = video.read()

    if not ret:
        break

    # YOLO + ByteTrack
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0],
        conf=0.4,
        verbose=False
    )

    # People currently visible
    current_people = set()

    boxes = results[0].boxes

    if boxes.id is not None:

        track_ids = boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes.xyxy, track_ids):

            # Give each tracker ID a simple number
            if track_id not in id_mapping:
                id_mapping[track_id] = next_person_id
                next_person_id += 1

            person_id = id_mapping[track_id]

            # Bounding box coordinates
            x1, y1, x2, y2 = map(
                int,
                box.cpu().tolist()
            )

            current_people.add(person_id)

            # Draw bounding box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Draw person ID
            cv2.putText(
                frame,
                f"Person ID: {person_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # Current number of people
    people_count = len(current_people)

    # Display counter
    cv2.putText(
        frame,
        f"People Tracked: {people_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # Save frame
    output.write(frame)

    frame_count += 1

video.release()
output.release()

print("================================")
print("TRACKING COMPLETED!")
print("================================")
print("Frames processed:", frame_count)
print("Unique IDs created:", next_person_id - 1)
print("Output file: final_people_tracking.mp4")

In [ ]:
from google.colab import files

files.download("final_people_tracking.mp4")

In [ ]:
%%writefile requirements.txt

ultralytics
opencv-python

In [ ]:
%%writefile README.md

# People Detection and Tracking

A computer vision project that detects and tracks people in video using YOLO, ByteTrack, and OpenCV.

## Features

- Detect people in video frames
- Assign unique tracking IDs
- Maintain IDs across video frames
- Draw bounding boxes around people
- Display the number of currently tracked people
- Save the processed video

## Technologies Used

- Python
- YOLO
- ByteTrack
- OpenCV
- Google Colab

## How It Works

The input video is processed frame-by-frame.

YOLO detects people in each frame. ByteTrack associates detections across consecutive frames and assigns tracking IDs.

The system then draws bounding boxes and tracking IDs on the video and displays the current number of tracked people.

## Project Pipeline

Video
↓
YOLO Person Detection
↓
ByteTrack
↓
Person Tracking IDs
↓
Bounding Boxes + People Count
↓
Output Video

## Files

- `People_Detection_Tracking.ipynb` - Google Colab notebook
- `requirements.txt` - Python dependencies
- `README.md` - Project documentation

## Future Improvements

- Improve ID consistency during occlusion
- Real-time webcam tracking
- Entry/exit counting
- Track people across multiple cameras